In [14]:
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(r"C:\Users\38760\Desktop\Unstructured_Data\News_Media_Monitoring_Pipeline_UD")

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from src.utils.logger import logging
except Exception:
    import logging
    logging.basicConfig(
        filename="pipeline.log",
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
    )

RAW_NEWS_PATH = Path("data/processed/analytics/raw_news_data.csv")
CLEANED_DIR = Path("data/processed/cleaned")
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw Lab 8 CSV exists:", RAW_NEWS_PATH.exists())
print("Cleaned output directory:", CLEANED_DIR)

logging.info("Lab 9 notebook started")

Project root: c:\Users\38760\Desktop\Unstructured_Data\News_Media_Monitoring_Pipeline_UD
Raw Lab 8 CSV exists: True
Cleaned output directory: data\processed\cleaned


In [15]:
from src.analytics.data_loader import load_from_mongodb, save_to_csv

if RAW_NEWS_PATH.exists():
    raw_df = pd.read_csv(RAW_NEWS_PATH)
    print("Loaded raw Lab 8 CSV:", RAW_NEWS_PATH)
else:
    print("Raw Lab 8 CSV not found. Loading from MongoDB and recreating it...")
    raw_df = load_from_mongodb()
    save_to_csv(raw_df, str(RAW_NEWS_PATH))

print("Raw dataset shape:", raw_df.shape)
display(raw_df.head())

logging.info("Lab 9 notebook loaded raw dataset with shape=%s", raw_df.shape)

Loaded raw Lab 8 CSV: data\processed\analytics\raw_news_data.csv
Raw dataset shape: (1455, 60)


,record_id,source_name,author,title,description,url,publishedAt,source_path,fetched_at,version,...,title_length,rating_score,overview,genres,popularity,release_date,release_year,original_language,vote_average,vote_count
0,1,Motley Fool Australia,James Mickleboro,"Why EOS, Humm, New Hope, and Sims shares are s...",These shares are having a good session on hump...,https://www.fool.com.au/2026/03/18/why-eos-hum...,2026-03-18T02:40:46Z,NaN,NaN,NaN,...,66,1.81,These shares are having a good session on hump...,news_api,181.0,NaN,NaN,unknown,1.81,1.0
1,2,The Punch,Punch Newspapers,SWDC partners FTID to boost S’West rural devt,The South-West Development Commission (SWDC) a...,https://punchng.com/swdc-partners-ftid-to-boos...,2026-03-18T02:34:47Z,NaN,NaN,NaN,...,45,2.26,The South-West Development Commission (SWDC) a...,news_api,226.0,NaN,NaN,unknown,2.26,1.0
2,3,The Times of India,TOI Education,"GATE 2026 final answer key not released yet, r...",The Indian Institute of Technology Guwahati ha...,https://timesofindia.indiatimes.com/education/...,2026-03-18T02:31:14Z,NaN,NaN,NaN,...,98,2.60,The Indian Institute of Technology Guwahati ha...,news_api,260.0,NaN,NaN,unknown,2.60,1.0
3,4,Financial Post,Business Wire,LTM Named NVIDIA Partner Network ‘Rising Star ...,"MUMBAI, India — LTM, the Business Creativity p...",https://financialpost.com/pmn/business-wire-ne...,2026-03-18T02:30:15Z,NaN,NaN,NaN,...,96,2.60,"MUMBAI, India — LTM, the Business Creativity p...",news_api,260.0,NaN,NaN,unknown,2.60,1.0
4,5,CNA,NaN,"Asian stocks rally as oil retreats, Fed in spo...","SYDNEY, March 18 : Asian shares rallied on Wed...",https://www.channelnewsasia.com/business/asian...,2026-03-18T02:27:11Z,NaN,NaN,NaN,...,52,2.60,"SYDNEY, March 18 : Asian shares rallied on Wed...",news_api,260.0,NaN,NaN,unknown,2.60,1.0


In [16]:
from src.cleaning.missing_handler import report_missing

missing_report = report_missing(raw_df)

missing_report_path = CLEANED_DIR / "missing_report.csv"
missing_report.to_csv(missing_report_path, index=False)

print("Missing value report:")
display(missing_report.head(30))
print("Saved missing report to:", missing_report_path)

logging.info("Lab 9 notebook missing value report saved to %s", missing_report_path)

Missing value report:


,column,missing_count,missing_pct,dtype
0,underline,1455,100.00,float64
1,preview_text,1448,99.52,str
2,Total Articles,1446,99.38,float64
3,Metric,1446,99.38,str
4,Politics Articles,1446,99.38,float64
5,Business Articles,1446,99.38,float64
6,Average Sentiment,1446,99.38,float64
7,Highest Mentions,1446,99.38,float64
8,Total Mentions,1446,99.38,float64
9,Average Mentions,1446,99.38,float64


Saved missing report to: data\processed\cleaned\missing_report.csv


In [17]:
from src.cleaning.missing_handler import (
    drop_rows_missing_critical_fields,
    fill_missing_text_fields,
    fill_missing_overview,
    replace_zero_with_nan,
    fill_numeric_with_median,
    drop_high_missingness_columns,
)

missing_handled_df = raw_df.copy()
rows_before = len(missing_handled_df)

missing_handled_df = drop_rows_missing_critical_fields(
    missing_handled_df,
    critical_columns=["record_id", "title"],
)

missing_handled_df = fill_missing_text_fields(missing_handled_df)
missing_handled_df = fill_missing_overview(missing_handled_df)

missing_handled_df = replace_zero_with_nan(
    missing_handled_df,
    columns=[
        "mentions",
        "sentiment_score",
        "rating_score",
        "popularity",
        "vote_average",
        "vote_count",
    ],
)

missing_handled_df = fill_numeric_with_median(
    missing_handled_df,
    columns=[
        "rating_score",
        "content_length",
        "title_length",
        "popularity",
        "vote_average",
        "vote_count",
    ],
)

missing_handled_df = drop_high_missingness_columns(
    missing_handled_df,
    threshold=0.60,
)

print("Rows before missing handling:", rows_before)
print("Rows after missing handling:", len(missing_handled_df))
print("Shape after missing handling:", missing_handled_df.shape)
display(missing_handled_df.head())

logging.info(
    "Lab 9 notebook missing handling complete: before=%d after=%d shape=%s",
    rows_before,
    len(missing_handled_df),
    missing_handled_df.shape,
)

Rows before missing handling: 1455
Rows after missing handling: 1455
Shape after missing handling: (1455, 34)


,record_id,source_name,title,description,url,source_path,fetched_at,version,document_type,file_name,...,published_year,content_length,title_length,rating_score,overview,genres,popularity,original_language,vote_average,vote_count
0,1,Motley Fool Australia,"Why EOS, Humm, New Hope, and Sims shares are s...",These shares are having a good session on hump...,https://www.fool.com.au/2026/03/18/why-eos-hum...,NaN,NaN,NaN,news_api,NaN,...,NaN,181,66,1.81,These shares are having a good session on hump...,news_api,181.0,unknown,1.81,1.0
1,2,The Punch,SWDC partners FTID to boost S’West rural devt,The South-West Development Commission (SWDC) a...,https://punchng.com/swdc-partners-ftid-to-boos...,NaN,NaN,NaN,news_api,NaN,...,NaN,226,45,2.26,The South-West Development Commission (SWDC) a...,news_api,226.0,unknown,2.26,1.0
2,3,The Times of India,"GATE 2026 final answer key not released yet, r...",The Indian Institute of Technology Guwahati ha...,https://timesofindia.indiatimes.com/education/...,NaN,NaN,NaN,news_api,NaN,...,NaN,260,98,2.60,The Indian Institute of Technology Guwahati ha...,news_api,260.0,unknown,2.60,1.0
3,4,Financial Post,LTM Named NVIDIA Partner Network ‘Rising Star ...,"MUMBAI, India — LTM, the Business Creativity p...",https://financialpost.com/pmn/business-wire-ne...,NaN,NaN,NaN,news_api,NaN,...,NaN,260,96,2.60,"MUMBAI, India — LTM, the Business Creativity p...",news_api,260.0,unknown,2.60,1.0
4,5,CNA,"Asian stocks rally as oil retreats, Fed in spo...","SYDNEY, March 18 : Asian shares rallied on Wed...",https://www.channelnewsasia.com/business/asian...,NaN,NaN,NaN,news_api,NaN,...,NaN,260,52,2.60,"SYDNEY, March 18 : Asian shares rallied on Wed...",news_api,260.0,unknown,2.60,1.0


In [18]:
from src.cleaning.string_cleaner import (
    clean_title,
    clean_language_code,
    clean_overview_text,
    extract_year_from_release_date,
    clean_genre_string,
    clean_string_columns,
)

string_clean_df = raw_df.copy()

string_clean_df = clean_title(string_clean_df)
string_clean_df = clean_language_code(string_clean_df)
string_clean_df = clean_overview_text(string_clean_df)
string_clean_df = extract_year_from_release_date(string_clean_df)
string_clean_df = clean_genre_string(string_clean_df)

preview_cols = [
    col for col in [
        "title",
        "language",
        "original_language",
        "category",
        "document_type",
        "content_text",
        "overview",
        "published_date",
        "published_year",
        "release_year",
    ]
    if col in string_clean_df.columns
]

print("String-cleaned preview:")
display(string_clean_df[preview_cols].head())

logging.info("Lab 9 notebook string cleaning complete")

String-cleaned preview:


,title,language,original_language,category,document_type,content_text,overview,published_date,published_year,release_year
0,"Why EOS, Humm, New Hope, and Sims shares are s...",unknown,unknown,news_api,news_api,These shares are having a good session on hump...,These shares are having a good session on hump...,NaN,<NA>,<NA>
1,SWDC partners FTID to boost S’West rural devt,unknown,unknown,news_api,news_api,The South-West Development Commission (SWDC) a...,The South-West Development Commission (SWDC) a...,NaN,<NA>,<NA>
2,"GATE 2026 final answer key not released yet, r...",unknown,unknown,news_api,news_api,The Indian Institute of Technology Guwahati ha...,The Indian Institute of Technology Guwahati ha...,NaN,<NA>,<NA>
3,LTM Named NVIDIA Partner Network ‘Rising Star ...,unknown,unknown,news_api,news_api,"MUMBAI, India — LTM, the Business Creativity p...","MUMBAI, India — LTM, the Business Creativity p...",NaN,<NA>,<NA>
4,"Asian stocks rally as oil retreats, Fed in spo...",unknown,unknown,news_api,news_api,"SYDNEY, March 18 : Asian shares rallied on Wed...","SYDNEY, March 18 : Asian shares rallied on Wed...",NaN,<NA>,<NA>


In [19]:
from src.analytics.regex_ops import (
    detect_invalid_date_formats,
    detect_invalid_language_codes,
    add_extracted_number_column,
    flag_short_content,
)

regex_df = string_clean_df.copy()

invalid_dates = detect_invalid_date_formats(regex_df, date_col="published_date")
invalid_languages = detect_invalid_language_codes(regex_df, language_col="language")

regex_df = add_extracted_number_column(
    regex_df,
    source_col="title",
    output_col="title_number_lab9",
)

regex_df = flag_short_content(
    regex_df,
    text_col="content_text",
    min_chars=40,
)

invalid_dates.to_csv(CLEANED_DIR / "regex_invalid_dates.csv", index=False)
invalid_languages.to_csv(CLEANED_DIR / "regex_invalid_languages.csv", index=False)
regex_df[["title", "title_number_lab9", "is_short_content"]].to_csv(
    CLEANED_DIR / "regex_title_numbers_and_short_content.csv",
    index=False,
)

print("Invalid date formats:")
display(invalid_dates.head())

print("Invalid language codes:")
display(invalid_languages.head())

print("Regex processed preview:")
display(regex_df[["title", "title_number_lab9", "is_short_content"]].head(10))

logging.info("Lab 9 notebook regex cleaning helpers complete")

Invalid date formats:


,row_index,column,value,issue


Invalid language codes:


,row_index,column,value,issue


Regex processed preview:


,title,title_number_lab9,is_short_content
0,"Why EOS, Humm, New Hope, and Sims shares are s...",NaN,False
1,SWDC partners FTID to boost S’West rural devt,NaN,False
2,"GATE 2026 final answer key not released yet, r...",2026,False
3,LTM Named NVIDIA Partner Network ‘Rising Star ...,2026,False
4,"Asian stocks rally as oil retreats, Fed in spo...",NaN,False
5,Trump administration defends Anthropic blackli...,NaN,False
6,Economic Digest: Nepal’s Business News in a Snap,NaN,False
7,"17 stocks including Tata Steel, SBI, Varun Bev...",17,False
8,Nvidia preparing Groq chips that can be sold i...,NaN,False
9,Heroes of Science and Fiction-TENOKE,NaN,False


In [20]:
from src.cleaning.deduplicator import (
    drop_exact_duplicates,
    drop_duplicate_ids,
    drop_duplicate_urls,
    drop_duplicate_titles,
    count_duplicates,
)

dedup_df = regex_df.copy()
dedup_summary = []

def add_dedup_step(step_name, before_rows, after_rows):
    dedup_summary.append({
        "step": step_name,
        "rows_before": before_rows,
        "rows_after": after_rows,
        "rows_removed": before_rows - after_rows,
    })

before = len(dedup_df)
dedup_df = drop_exact_duplicates(dedup_df)
add_dedup_step("drop_exact_duplicates", before, len(dedup_df))

duplicate_record_ids_before = count_duplicates(dedup_df, "record_id") if "record_id" in dedup_df.columns else 0
duplicate_urls_before = count_duplicates(dedup_df.dropna(subset=["url"]), "url") if "url" in dedup_df.columns else 0

before = len(dedup_df)
dedup_df = drop_duplicate_urls(dedup_df)
add_dedup_step("drop_duplicate_urls", before, len(dedup_df))

before = len(dedup_df)
dedup_df = drop_duplicate_titles(dedup_df)
add_dedup_step("drop_duplicate_titles", before, len(dedup_df))

before = len(dedup_df)
dedup_df = drop_duplicate_ids(dedup_df, id_col="record_id")
add_dedup_step("drop_duplicate_ids", before, len(dedup_df))

duplicate_record_ids_after = count_duplicates(dedup_df, "record_id") if "record_id" in dedup_df.columns else 0
duplicate_urls_after = count_duplicates(dedup_df.dropna(subset=["url"]), "url") if "url" in dedup_df.columns else 0

dedup_summary_df = pd.DataFrame(dedup_summary)
dedup_summary_df.to_csv(CLEANED_DIR / "deduplication_summary.csv", index=False)

print("Duplicate record IDs before:", duplicate_record_ids_before)
print("Duplicate record IDs after:", duplicate_record_ids_after)
print("Duplicate URLs before:", duplicate_urls_before)
print("Duplicate URLs after:", duplicate_urls_after)

display(dedup_summary_df)
print("Shape after deduplication:", dedup_df.shape)

logging.info("Lab 9 notebook deduplication complete: shape=%s", dedup_df.shape)

Duplicate record IDs before: 0
Duplicate record IDs after: 0
Duplicate URLs before: 57
Duplicate URLs after: 0


,step,rows_before,rows_after,rows_removed
0,drop_exact_duplicates,1455,1455,0
1,drop_duplicate_urls,1455,1398,57
2,drop_duplicate_titles,1398,1318,80
3,drop_duplicate_ids,1318,1318,0


Shape after deduplication: (1318, 62)


In [21]:
from src.cleaning.type_converter import (
    convert_dates,
    convert_numeric_columns,
    convert_category_columns,
    memory_report,
    convert_all_types,
)

type_before_df = dedup_df.copy()
typed_df = dedup_df.copy()

typed_df = convert_dates(typed_df)
typed_df = convert_numeric_columns(typed_df)
typed_df = convert_category_columns(typed_df)

memory_stats = memory_report(type_before_df, typed_df)

memory_report_df = pd.DataFrame([memory_stats])
memory_report_df.to_csv(CLEANED_DIR / "type_conversion_memory_report.csv", index=False)

print("Converted dtypes:")
display(
    pd.DataFrame({
        "column": typed_df.columns,
        "dtype": [str(dtype) for dtype in typed_df.dtypes],
    }).head(40)
)

print("Memory report:")
display(memory_report_df)

logging.info("Lab 9 notebook type conversion complete")

Memory before: 3.29 MB
Memory after:  2.25 MB
Saved:         1.04 MB  (31.6%)
Converted dtypes:


,column,dtype
0,record_id,Int64
1,source_name,category
2,author,str
3,title,str
4,description,str
5,url,str
6,publishedAt,"datetime64[us, UTC]"
7,source_path,str
8,fetched_at,datetime64[us]
9,version,float64


Memory report:


,before_mb,after_mb,saved_mb,saved_pct
0,3.291237,2.249908,1.041328,31.639425


In [22]:
from src.cleaning.validator import run_all_validations

validation_result = run_all_validations(typed_df)

validation_summary_df = pd.DataFrame([validation_result])
validation_summary_df.to_csv(CLEANED_DIR / "validation_summary.csv", index=False)

display(validation_summary_df)

logging.info("Lab 9 notebook validation complete: %s", validation_result)

  PASSED: validate_required_columns
  PASSED: validate_no_null_titles
  PASSED: validate_rating_score_range
  PASSED: validate_year_range
  PASSED: validate_no_duplicate_ids
  PASSED: validate_language_codes
  PASSED: validate_content_length

Validation complete: 7 passed, 0 failed


,passed,failed,failures
0,7,0,[]


In [23]:
from src.cleaning.clean_pipeline import run_cleaning_pipeline

cleaned_df = run_cleaning_pipeline(raw_df, save=True)

print("Final cleaned dataset shape:", cleaned_df.shape)
display(cleaned_df.head())

print("Saved cleaned files:")
for path in [
    CLEANED_DIR / "missing_report.csv",
    CLEANED_DIR / "cleaned_data.csv",
    CLEANED_DIR / "clean.csv",
]:
    print(path, "->", path.exists())

logging.info("Lab 9 notebook full reusable pipeline complete: shape=%s", cleaned_df.shape)

Memory before: 2.71 MB
Memory after:  1.91 MB
Saved:         0.79 MB  (29.3%)
  PASSED: validate_required_columns
  PASSED: validate_no_null_titles
  PASSED: validate_rating_score_range
  PASSED: validate_year_range
  PASSED: validate_no_duplicate_ids
  PASSED: validate_language_codes
  PASSED: validate_content_length

Validation complete: 7 passed, 0 failed
Final cleaned dataset shape: (1318, 35)


,record_id,source_name,title,description,url,source_path,fetched_at,version,document_type,file_name,...,published_year,content_length,title_length,rating_score,overview,genres,popularity,original_language,vote_average,vote_count
0,52,bbc,Election Update,No content available.,NaN,NaN,NaT,NaN,excel,news_data.xlsx,...,2026,15,15,4.0,Election Update,politics,34.0,unknown,4.0,34.0
1,53,reuters,AI Market Growth,No content available.,NaN,NaN,NaT,NaN,excel,news_data.xlsx,...,2026,16,16,7.0,AI Market Growth,business,28.0,unknown,7.0,28.0
2,54,cnn,Sports Highlights,No content available.,NaN,NaN,NaT,NaN,excel,news_data.xlsx,...,2026,17,17,5.0,Sports Highlights,sports,19.0,unknown,5.0,19.0
3,55,ap,Climate Policy Shift,No content available.,NaN,NaN,NaT,NaN,excel,news_data.xlsx,...,2026,20,20,3.0,Climate Policy Shift,politics,23.0,unknown,3.0,23.0
4,56,le monde,Café Culture Trends,No content available.,NaN,NaN,NaT,NaN,excel,news_data.xlsx,...,2026,19,19,6.0,Café Culture Trends,culture,14.0,unknown,6.0,14.0


Saved cleaned files:
data\processed\cleaned\missing_report.csv -> True
data\processed\cleaned\cleaned_data.csv -> True
data\processed\cleaned\clean.csv -> True


In [24]:
import subprocess

pytest_cmd = [sys.executable, "-m", "pytest", "tests/test_cleaning.py", "-v"]

pytest_result = subprocess.run(
    pytest_cmd,
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

pytest_output = pytest_result.stdout + "\n" + pytest_result.stderr

pytest_output_path = CLEANED_DIR / "pytest_output.txt"
pytest_output_path.write_text(pytest_output, encoding="utf-8")

print(pytest_output)
print("Pytest return code:", pytest_result.returncode)
print("Saved pytest output to:", pytest_output_path)

assert pytest_result.returncode == 0, "Pytest cleaning tests failed"

logging.info("Lab 9 notebook pytest complete with return code %s", pytest_result.returncode)

============================= test session starts =============================
platform win32 -- Python 3.14.3, pytest-9.0.3, pluggy-1.6.0 -- c:\Users\38760\Desktop\Unstructured_Data\News_Media_Monitoring_Pipeline_UD\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\38760\Desktop\Unstructured_Data\News_Media_Monitoring_Pipeline_UD
plugins: anyio-4.13.0, cov-7.1.0
collecting ... collected 26 items

tests/test_cleaning.py::test_report_missing_detects_missing_values PASSED [  3%]
tests/test_cleaning.py::test_drop_rows_missing_title_removes_empty_string PASSED [  7%]
tests/test_cleaning.py::test_fill_missing_overview_no_nulls_remain PASSED [ 11%]
tests/test_cleaning.py::test_fill_missing_overview_uses_placeholder PASSED [ 15%]
tests/test_cleaning.py::test_replace_zero_with_nan_on_rating_score PASSED [ 19%]
tests/test_cleaning.py::test_fill_numeric_with_median PASSED             [ 23%]
tests/test_cleaning.py::test_drop_high_missingness_columns PASSED        [ 26%]
tests/te

In [25]:
expected_outputs = [
    CLEANED_DIR / "missing_report.csv",
    CLEANED_DIR / "cleaned_data.csv",
    CLEANED_DIR / "clean.csv",
    CLEANED_DIR / "deduplication_summary.csv",
    CLEANED_DIR / "type_conversion_memory_report.csv",
    CLEANED_DIR / "validation_summary.csv",
    CLEANED_DIR / "pytest_output.txt",
    CLEANED_DIR / "regex_invalid_dates.csv",
    CLEANED_DIR / "regex_invalid_languages.csv",
    CLEANED_DIR / "regex_title_numbers_and_short_content.csv",
]

verification_df = pd.DataFrame({
    "path": [str(path) for path in expected_outputs],
    "exists": [path.exists() for path in expected_outputs],
})

display(verification_df)

assert verification_df["exists"].all(), "Some Lab 9 outputs are missing"

logging.info("Lab 9 notebook final verification complete")

,path,exists
0,data\processed\cleaned\missing_report.csv,True
1,data\processed\cleaned\cleaned_data.csv,True
2,data\processed\cleaned\clean.csv,True
3,data\processed\cleaned\deduplication_summary.csv,True
4,data\processed\cleaned\type_conversion_memory_...,True
5,data\processed\cleaned\validation_summary.csv,True
6,data\processed\cleaned\pytest_output.txt,True
7,data\processed\cleaned\regex_invalid_dates.csv,True
8,data\processed\cleaned\regex_invalid_languages...,True
9,data\processed\cleaned\regex_title_numbers_and...,True
